# Lab 8 - Concurrent orchestration

## What will you do?

Lab 7 ran agents one after another. Here you use Agent Framework's **`ConcurrentBuilder`** to run three specialist agents **at the same time** on the same question, then compare their independent opinions.

![Diagram that shows concurrent orchestration where multiple agents process the same input task simultaneously and their results are aggregated.](https://learn.microsoft.com/en-us/azure/architecture/ai-ml/guide/_images/concurrent-pattern.svg)

*Concurrent orchestration. Source: [AI Agent Orchestration Patterns](https://learn.microsoft.com/en-us/azure/architecture/ai-ml/guide/ai-agent-design-patterns#concurrent-orchestration) on Microsoft Learn.*

```text
                 -> cardiology agent    \
question         -> primary-care agent   -> one response, one message per agent
                 -> public-health agent  /
```

No agent sees another's answer. There is nothing to merge into a single recommendation here; you are comparing perspectives, not voting on one answer.

> **This is a workshop exercise, not a clinical tool.**

## Before you start

- Python 3.11 or later, with a notebook kernel selected, and `az login` completed.
- Lab 1 finished: a project endpoint and an approved model deployment.
- No knowledge base or tools are needed for this lab.

Install the pinned package set below. If you already imported a different version of these SDKs in this kernel, restart the kernel after installing.

In [ ]:
%pip install -q "azure-ai-projects==2.3.0" "azure-identity==1.25.3" "openai==2.54.0" "agent-framework-core==1.16.0" "agent-framework-openai==1.14.1" "agent-framework-foundry==1.11.0" "agent-framework-orchestrations==1.1.1"

## 0. Connect to your project

Use your Lab 1 settings. The cell signs in with your Azure CLI account and generates a suffix to keep your agents unique in the shared project.

In [ ]:
import asyncio
import os
import sys
from contextlib import AsyncExitStack
from uuid import uuid4

from agent_framework.foundry import FoundryAgent
from agent_framework.orchestrations import ConcurrentBuilder
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition
from azure.identity import AzureCliCredential

# Your nonsecret settings from Lab 1. Paste them between the quotes, or set them as
# environment variables before starting the kernel.
PROJECT_ENDPOINT = os.getenv("AZURE_AI_PROJECT_ENDPOINT", "")
MODEL_DEPLOYMENT = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME", "")

if sys.version_info < (3, 11):
    raise RuntimeError(
        "These notebooks need Python 3.11 or later. This kernel is "
        f"{sys.version_info.major}.{sys.version_info.minor}. Select a newer kernel."
    )

missing = [
    name
    for name, value in {
        "AZURE_AI_PROJECT_ENDPOINT": PROJECT_ENDPOINT,
        "AZURE_AI_MODEL_DEPLOYMENT_NAME": MODEL_DEPLOYMENT,
    }.items()
    if not value
]
if missing:
    raise ValueError(f"Set these before continuing: {', '.join(missing)}")


def check_todos(**answers: object) -> None:
    """Helper. Stops the cell while a `...` blank is still open."""
    still_open = [name for name, value in answers.items() if value is ...]
    if still_open:
        raise ValueError(f"Fill in these blanks first: {', '.join(still_open)}")


SUFFIX = uuid4().hex[:8]
credential = AzureCliCredential()
project = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
print(f"Setup complete. Suffix: {SUFFIX}")

## 1. Register three specialist agents

Order does not matter here; all three run on the same input at once. What matters is that each instruction gives a genuinely different angle on the question.

### To-Do 1 - Write the three perspectives

**Goal:** three instructions that would visibly disagree on emphasis, not just repeat each other.

**Steps**

1. Write `CARDIOLOGY_INSTRUCTIONS`, `PRIMARY_CARE_INSTRUCTIONS` and `PUBLIC_HEALTH_INSTRUCTIONS`.
2. Run the cell.

**Run the cell. You should see** three agent names printed.

<details><summary>Hint</summary>

Give each agent a role and a narrow focus: cardiovascular risk, daily routine, or population-level guidance.

</details>

<details><summary>Show solution code</summary>

```python
CARDIOLOGY_INSTRUCTIONS = "You are a cardiologist. Comment on cardiovascular risk and monitoring."
PRIMARY_CARE_INSTRUCTIONS = "You are a primary care physician. Comment on how this fits daily routine and follow-up."
PUBLIC_HEALTH_INSTRUCTIONS = "You are a public health specialist. Comment on population-level guidance and prevention."
```

</details>

In [ ]:
CARDIOLOGY_INSTRUCTIONS = ...  # TODO 1: a cardiology perspective on a treatment question.
PRIMARY_CARE_INSTRUCTIONS = ...  # TODO 1: a primary-care perspective, focused on daily routine.
PUBLIC_HEALTH_INSTRUCTIONS = ...  # TODO 1: a public-health perspective, focused on population-level guidance.
check_todos(
    CARDIOLOGY_INSTRUCTIONS=CARDIOLOGY_INSTRUCTIONS,
    PRIMARY_CARE_INSTRUCTIONS=PRIMARY_CARE_INSTRUCTIONS,
    PUBLIC_HEALTH_INSTRUCTIONS=PUBLIC_HEALTH_INSTRUCTIONS,
)

cardiology_version = project.agents.create_version(
    agent_name=f"day2-cardiology-opinion-{SUFFIX}",
    definition=PromptAgentDefinition(model=MODEL_DEPLOYMENT, instructions=CARDIOLOGY_INSTRUCTIONS),
)
primary_care_version = project.agents.create_version(
    agent_name=f"day2-primary-care-opinion-{SUFFIX}",
    definition=PromptAgentDefinition(model=MODEL_DEPLOYMENT, instructions=PRIMARY_CARE_INSTRUCTIONS),
)
public_health_version = project.agents.create_version(
    agent_name=f"day2-public-health-opinion-{SUFFIX}",
    definition=PromptAgentDefinition(model=MODEL_DEPLOYMENT, instructions=PUBLIC_HEALTH_INSTRUCTIONS),
)
print("Registered:", cardiology_version.name, primary_care_version.name, public_health_version.name)

## 2. Build and run the concurrent workflow

`ConcurrentBuilder` runs every participant on the same input and combines the results into one `AgentResponse`, with one message per agent. The order of `participants` does not affect execution.

In [ ]:
QUESTION = (
    "A 58-year-old patient with well-controlled type 2 diabetes wants to start jogging three "
    "times a week. What should they consider?"
)

async with AsyncExitStack() as stack:
    cardiology = await stack.enter_async_context(
        FoundryAgent(
            project_endpoint=PROJECT_ENDPOINT,
            agent_name=cardiology_version.name,
            agent_version=str(cardiology_version.version),
            credential=credential,
            name="cardiology",
            allow_preview=False,
            timeout=240,
        )
    )
    primary_care = await stack.enter_async_context(
        FoundryAgent(
            project_endpoint=PROJECT_ENDPOINT,
            agent_name=primary_care_version.name,
            agent_version=str(primary_care_version.version),
            credential=credential,
            name="primary_care",
            allow_preview=False,
            timeout=240,
        )
    )
    public_health = await stack.enter_async_context(
        FoundryAgent(
            project_endpoint=PROJECT_ENDPOINT,
            agent_name=public_health_version.name,
            agent_version=str(public_health_version.version),
            credential=credential,
            name="public_health",
            allow_preview=False,
            timeout=240,
        )
    )

    workflow = ConcurrentBuilder(participants=[cardiology, primary_care, public_health]).build()
    events = await asyncio.wait_for(workflow.run(QUESTION), timeout=600)

outputs = events.get_outputs()
final = outputs[0]
for message in final.messages:
    print(f"[{message.author_name or 'agent'}]\n{message.text}\n")

## Deterministic success check

This checks the shape of the run, not the wording: three distinct agents each contributed one message.

In [ ]:
assert len({cardiology_version.name, primary_care_version.name, public_health_version.name}) == 3, (
    "The three agents should be distinct."
)
assert len(final.messages) == 3, f"Expected one message per agent, got {len(final.messages)}."
assert all(message.text.strip() for message in final.messages), "An agent returned no text."

authors = {message.author_name for message in final.messages}
assert len(authors) == 3, f"Expected three distinct authors, got {authors}."

print("PASS - three specialist agents ran concurrently and each contributed an independent opinion.")

## What you learned

1. `ConcurrentBuilder` fans the same input out to every participant and fans the results back in.
2. The default aggregation is one message per agent; nothing forces the opinions into agreement.
3. Concurrent orchestration reduces latency versus a pipeline, but only helps when the agents do not depend on each other's output.

**Further reading:** [Concurrent orchestration](https://learn.microsoft.com/en-us/agent-framework/workflows/orchestrations/concurrent) on Microsoft Learn, and [AI Agent Orchestration Patterns](https://learn.microsoft.com/en-us/azure/architecture/ai-ml/guide/ai-agent-design-patterns#concurrent-orchestration) for when to choose this pattern.

**Next:** Lab 9 tackles a task with no fixed order at all, using **Magentic** orchestration.

In [ ]:
project.close()
credential.close()
print("Closed the local clients. All Foundry agents remain.")